# Data Cleaning

In [1]:
import pandas as pd

df = pd.read_csv("C:\\Users\\MY PC\\Downloads\\matplotlib_messy_sales_2000 - matplotlib_messy_sales_2000.csv")
df.head(2)

,Customer ID,Order Date,City,Product Category,Customer Segment,Gender,Sales Channel,Age,Units Sold,Unit Price,Discount %,Customer Rating,Is Active,Returned,Payment Method,Gross Sales,Discount Amount,Revenue,Cost,Profit
0,CUST00594,2025-05-05 0:00:00,Mumbai,Clothing,premium,Male,Retail,36,8,24235.73,22.91,1.9,1,0,UPI,193885.84,44419.25,149466.59,88698.45,60768.14
1,CUST01103,2025-10-22 0:00:00,Mumbai,Clothing,Premium,Female,Corporate,38,5,16789.55,10.59,4.3,1,0,Cash,83947.75,8890.07,75057.68,61426.31,13631.37


# 1. Column Names
# Display all column names. Identify columns having leading/trailing spaces and
# rename all columns into a consistent snake_case format.

In [2]:
df.columns.tolist()

['Customer ID',
 'Order Date',
 'City',
 'Product Category',
 'Customer Segment',
 'Gender',
 'Sales Channel',
 'Age',
 'Units Sold',
 'Unit Price',
 'Discount %',
 'Customer Rating',
 'Is Active',
 'Returned',
 'Payment Method',
 'Gross Sales',
 'Discount Amount',
 'Revenue',
 'Cost',
 'Profit']

In [3]:
df.columns = (
    df.columns.str.strip().str.lower().str.replace(" ", "_")
)
df.columns

Index(['customer_id', 'order_date', 'city', 'product_category',
       'customer_segment', 'gender', 'sales_channel', 'age', 'units_sold',
       'unit_price', 'discount_%', 'customer_rating', 'is_active', 'returned',
       'payment_method', 'gross_sales', 'discount_amount', 'revenue', 'cost',
       'profit'],
      dtype='object')

# 2. Duplicate Records
# Find the number of duplicate rows. Remove the duplicate records and verify that no
# duplicates remain.

In [4]:
df.duplicated().sum()

np.int64(20)

In [17]:
df.drop_duplicates(inplace=True)

# 3. Missing Values
# Display the missing-value count and missing-value percentage for every column.

In [6]:
df.isnull().sum()

customer_id          0
order_date           4
city                25
product_category    24
customer_segment     0
gender               0
sales_channel        0
age                 27
units_sold           4
unit_price          24
discount_%          28
customer_rating     25
is_active            0
returned             0
payment_method      25
gross_sales          0
discount_amount      0
revenue             25
cost                 0
profit               0
dtype: int64

In [7]:
print((df.isnull().sum() / len(df) * 100).round(2))

customer_id         0.00
order_date          0.20
city                1.25
product_category    1.20
customer_segment    0.00
gender              0.00
sales_channel       0.00
age                 1.35
units_sold          0.20
unit_price          1.20
discount_%          1.40
customer_rating     1.25
is_active           0.00
returned            0.00
payment_method      1.25
gross_sales         0.00
discount_amount     0.00
revenue             1.25
cost                0.00
profit              0.00
dtype: float64


# 4. Missing Categorical Values
# Handle missing values in city, product_category, and payment_method using an
# appropriate strategy.

In [8]:
df['city'] = df['city'].fillna(df['city'].mode()[0])

df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])

df['payment_method'] = df['payment_method'].fillna(
    df['payment_method'].mode()[0]
)

In [9]:
print(df[['city', 'product_category', 'payment_method']].isna().sum())

city                0
product_category    0
payment_method      0
dtype: int64


# 5. Missing Numerical Values
# Handle missing values in age, unit_price, discount_pct, customer_rating, and
# revenue. Explain why you selected mean, median, or another method.

In [10]:
df['age'] = pd.to_numeric(df['age'],errors ='coerce')
df['age'] = df['age'].fillna(df['age'].median())

df['unit_price'] = pd.to_numeric(df['unit_price'],errors ='coerce')
df['unit_price'] = df['unit_price'].fillna(df['unit_price'].median())

df['discount_%'] = pd.to_numeric(df['discount_%'],errors ='coerce')
df['discount_%'] = df['discount_%'].fillna(df['discount_%'].median())

df['customer_rating'] = pd.to_numeric(df['customer_rating'],errors ='coerce')
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

df['revenue'] = pd.to_numeric(df['revenue'],errors ='coerce')
df['revenue'] = df['revenue'].fillna(df['revenue'].median())


In [11]:
print(df[['age', 'unit_price', 'discount_%','customer_rating','revenue']].isna().sum())

age                0
unit_price         0
discount_%         0
customer_rating    0
revenue            0
dtype: int64


# 6. Date Cleaning
# Convert order_date into a proper datetime format. Identify invalid date values and
# handle them appropriately.

In [12]:
df['order_date'] = pd.to_datetime(df['order_date'],errors ='coerce')

In [18]:
df['order_date'] = pd.to_datetime(df['order_date'],errors ='coerce')
df['order_date'].isna().sum()
print(df[df['order_date'].isna()])

df.dropna(subset=['order_date'], inplace=True)
df['order_date'].isna().sum()

Empty DataFrame
Columns: [customer_id, order_date, city, product_category, customer_segment, gender, sales_channel, age, units_sold, unit_price, discount_%, customer_rating, is_active, returned, payment_method, gross_sales, discount_amount, revenue, cost, profit]
Index: []


np.int64(0)

# 7. Age Cleaning
# The age column contains values such as 25 years, 30 yrs, 45Y, and N/A. Extract the
# numerical age and convert the column into a numeric datatype.

In [29]:

df['age'] = pd.to_numeric(df['age'],errors ='coerce')
df['age'].isna().sum()


np.int64(0)

# 8. Age Outliers
# Identify unrealistic ages such as 120, 150, and 200. Detect them using the IQR
# method and handle them appropriately.

In [15]:
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['age'] < lower) | (df['age'] > upper)]
print("Age outliers:")
print(outliers['age'])

median_age = df['age'].median()

df.loc[
    (df['age'] < lower) | (df['age'] > upper),
    'age'
] = median_age

print((df['age'] < lower) | (df['age'] > upper))

Age outliers:
193     150.0
256     120.0
323     120.0
467     120.0
534     200.0
647     120.0
777     150.0
869     150.0
991     150.0
1023    120.0
1532    150.0
1731    120.0
2000    120.0
Name: age, dtype: float64
0       False
1       False
2       False
3       False
4       False
        ...  
2015    False
2016    False
2017    False
2018    False
2019    False
Name: age, Length: 1975, dtype: bool


# 9. Unit Price Cleaning
# The unit_price column contains values such as ₹15000, $20000, 25000 INR, and 10k.
# Convert all valid values into a single numeric format.

In [28]:

df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

df['unit_price'].isna().sum()

np.int64(0)

# 10. Unit Price Outliers
# Detect extreme unit_price values using the IQR method. Compare the number of
# outliers before and after treatment.

In [22]:
Q1 = df['unit_price'].quantile(0.25)
Q3 = df['unit_price'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers_before = ((df['unit_price'] < lower) | (df['unit_price'] > upper)).sum()

print("Outliers before treatment:", outliers_before)

df['unit_price'] = df['unit_price'].clip(lower=lower, upper=upper)

outliers_after = ((df['unit_price'] < lower) | 
                  (df['unit_price'] > upper)).sum()

print("Outliers after treatment:", outliers_after)

Outliers before treatment: 0
Outliers after treatment: 0


# 11. Units Sold Cleaning
# Clean values such as 5 units, 10 pcs, twenty, N/A, and 3 and convert the column into
# a numeric datatype.

In [23]:
df['units_sold'].unique()

array(['8', '5', '2', '9', '1', '10', '12', '11', '4', '13', '7', '3',
       '6', '14', '10 pcs', '100', '5 units', '200', nan, 'twenty', '50'],
      dtype=object)

In [27]:
df['units_sold'] = pd.to_numeric(df['units_sold'], errors='coerce')
df['units_sold'].isna().sum()

np.int64(4)

# 12. Discount Cleaning
# Clean values such as 10%, 20 percent, 5 %, none, and N/A. Convert the final
# discount_pct column into numeric values between 0 and 100.

In [32]:
df['discount_%'] = pd.to_numeric(df['discount_%'], errors='coerce')

df['discount_%'].isna().sum()

np.int64(0)